Run before everything

In [2]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

192.168.88.1:50051


Part1. ZIGGIE

In [ ]:
import time
import math

# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
CENTER_X = 320
GAP_TOLERANCE = 10    # Relaxed tolerance for initial alignment
STRICT_TOLERANCE = 2.5  # Strict tolerance for the final grab position
TARGET_DISTANCE = 0.16  # Distance to stop for the grab
RED_GROUP = ["Red", "Orange", "Yellow"]
GREEN_GROUP = ["Green", "Lime", "Cyan"]

# ==========================================
# HELPERS
# ==========================================
def safe_get_tag():
    """Returns tag info if 1 or more tags are seen, else None."""
    try:
        data = got.get_apriltag_total_info()
        if data and len(data) > 0:
            return data
    except:
        return None
    return None

def is_target_color(detected, target):
    if not detected:
        return False
    if target == "Red":
        return detected[0] in RED_GROUP
    if target == "Green":
        return detected[0] in GREEN_GROUP
    return False

def wait_for_color(target):
    print(f"[STATE] Waiting for {target} color card...")
    while True:
        color = got.get_color_total_info()
        if is_target_color(color, target):
            print(f"[STATE] {target} detected!")
            return True
        time.sleep(0.05)

def show_color_screen(color_name):
    """Displays specific background color on UGOT screen for 2 seconds."""
    if color_name == "Red":
        got.screen_display_background(3)
    elif color_name == "Green":
        got.screen_display_background(6)
    time.sleep(2)
    got.screen_display_background(0)

# ==========================================
# ADVANCED HUNT (WIDE ZIG-ZAG + CORNER LOCK)
# ==========================================
def AP_advanced_hunt(fwd_speed=3, strafe_speed=8):
    print("[MISSION] Starting Wide Zig-Zag with Corner Lock...")
    
    half_time = 3 
    full_time = 6
    start_time = time.time()
    
    # 1. SEARCH & LOCK PHASE
    while True:
        AP_info = safe_get_tag()
        
        if AP_info:
            tag_x = AP_info[0][1]
            # --- CORNER LOCK LOGIC ---
            if tag_x < 245:
                got.mecanum_move_xyz(-10, fwd_speed, 0)
                continue
            elif tag_x > 395:
                got.mecanum_move_xyz(10, fwd_speed, 0)
                continue
            else:
                got.mecanum_stop()
                print("[FOUND] Tag in vicinity. Adjusting...")
                time.sleep(0.3) 
                break
        
        elapsed = time.time() - start_time
        if elapsed < half_time:
            current_strafe = strafe_speed
        elif (elapsed - half_time) % (full_time * 2) < full_time:
            current_strafe = -strafe_speed
        else:
            current_strafe = strafe_speed

        got.mecanum_move_xyz(int(current_strafe), fwd_speed, 0)
        time.sleep(0.02)

    # 2. ALIGNMENT & APPROACH PHASE (Dynamic Tolerance)
    print("[ALIGN] Centralizing and Approaching...")
    while True:
        AP_info = safe_get_tag()
        if not AP_info:
            got.mecanum_move_xyz(4, 0, 0) 
            time.sleep(0.1)
            continue

        AP_x = AP_info[0][1]
        dist = AP_info[0][6]
        error_x = AP_x - CENTER_X

        # Decide which tolerance to use
        # If we are close to the target distance, use strict (2px) tolerance
        current_tol = STRICT_TOLERANCE if dist < (TARGET_DISTANCE + 0.05) else GAP_TOLERANCE

        # Horizontal Adjustment
        if abs(error_x) > current_tol:
            side_move = 5 if error_x > 0 else -5
            # Slow down forward if we need to adjust side-to-side strictly
            fwd_move = 2 if dist > TARGET_DISTANCE else 0
            got.mecanum_move_xyz(int(side_move), fwd_move, 0)
        else:
            # If centered within current tolerance, move forward
            if dist > TARGET_DISTANCE:
                got.mecanum_move_xyz(0, 5, 0)
            else:
                got.mecanum_stop()
                print(f"[READY] Centered within {current_tol}px. Grabbing.")
                break
        
        time.sleep(0.05)

# ==========================================
# PICKUP SEQUENCE
# ==========================================
def advanced_pick_up():
    print("[PICKUP] Executing Grab Sequence...")
    
    got.mechanical_single_joint_control(joint=2, angle=15, duration=500)
    got.mechanical_single_joint_control(joint=3, angle=-75, duration=500)
    got.mechanical_clamp_release()
    time.sleep(1.0)

    AP_info = safe_get_tag()
    if not AP_info:
        joint1, joint3 = 0, 10
    else:
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]
        joint1 = int(max(min((AP_x - CENTER_X) * -0.12, 90), -90))
        joint3 = int(max(min(AP_distance * 100 - 85, 90), -90))

    got.mechanical_joint_control(joint1, 0, joint3, 800)
    time.sleep(1)
    
    got.mecanum_translate_speed_times(angle=0, speed=2, times=2, unit=1)
    time.sleep(1)
    
    got.mechanical_clamp_close()
    time.sleep(1.0)
    
    got.mechanical_joint_control(0, 30, 30, 1000)
    print("[SUCCESS] Mission Complete.")

# ==========================================
# MAIN EXECUTION FLOW
# ==========================================
def run_autonomous_mission():
    try:
        wait_for_color("Red")
        show_color_screen("Red")
        wait_for_color("Green")
        show_color_screen("Green")
        AP_advanced_hunt()
        advanced_pick_up()
    except KeyboardInterrupt:
        got.mecanum_stop()
    except Exception as e:
        print(f"Error: {e}")
        got.mecanum_stop()

if __name__ == "__main__":
    run_autonomous_mission()

[STATE] Waiting for Red color card...
[STATE] Red detected!
[STATE] Waiting for Green color card...
[STATE] Green detected!
[MISSION] Starting Wide Zig-Zag with Corner Lock...
Error: 'float' object cannot be interpreted as an integer
